In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import os

### Joining products and order items csv's to the master sheet

In [2]:
df_order_items = pd.read_csv(os.path.join("../Data", "olist_order_items_dataset.csv"))
df_products    = pd.read_csv(os.path.join("../Data", "olist_products_dataset.csv"))

# Keep one product per order to avoid duplicates
df_order_items_clean = (
    df_order_items[["order_id", "product_id"]]
    .drop_duplicates("order_id", keep="first")
)


df = pd.read_csv(os.path.join("../Merged_Olist_Data", "delivery_analysis.csv"))

# Join order items then products
df = df.merge(df_order_items_clean, on="order_id", how="left")
df = df.merge(df_products[["product_id", "product_category_name"]], on="product_id", how="left")

# Verify row count
print(f"Row count: {len(df)}")
print(f"Product category sample:\n{df['product_category_name'].value_counts().head(5)}")

# Save updated file
df.to_csv(os.path.join("../Merged_Olist_Data", "delivery_analysis.csv"), index=False)
print("delivery_analysis.csv updated with product categories")

Row count: 98207
Product category sample:
product_category_name
cama_mesa_banho           9293
beleza_saude              8761
esporte_lazer             7634
informatica_acessorios    6625
moveis_decoracao          6331
Name: count, dtype: int64
delivery_analysis.csv updated with product categories


### Load Product Category Translation Table

In [3]:
# Load translation file directly from Kaggle data
df_translation = pd.read_csv(os.path.join("../Data", "product_category_name_translation.csv"))

print(f"Total categories translated: {len(df_translation)}")
print(f"\nColumns: {df_translation.columns.tolist()}")
df_translation.head()

Total categories translated: 71

Columns: ['product_category_name', 'product_category_name_english']


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


### Load Delivery Analysis and Join Translation

In [4]:
df = pd.read_csv(os.path.join("../Merged_Olist_Data", "delivery_analysis.csv"))

# Join translation
df = df.merge(df_translation, on="product_category_name", how="left")

print(f"Shape after join: {df.shape}")
print(f"\nSample of English categories:")
print(df["product_category_name_english"].value_counts().head(10))

Shape after join: (98207, 23)

Sample of English categories:
product_category_name_english
bed_bath_table           9293
health_beauty            8761
sports_leisure           7634
computers_accessories    6625
furniture_decor          6331
housewares               5792
watches_gifts            5581
telephony                4166
auto                     3855
toys                     3830
Name: count, dtype: int64


In [5]:
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_id,review_score,...,review_answer_timestamp,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,Days_Difference,Delivery_Status,product_id,product_category_name,product_category_name_english
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,a54f0611adc9ed256b57ede6b6eb5114,4.0,...,2017-10-12 03:43:48,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,7.0,On Time,87285b34884572647811a353c7ac498a,utilidades_domesticas,housewares
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,8d5266042046a06655c8db133d120ba5,4.0,...,2018-08-08 18:37:50,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,5.0,On Time,595fac2a385ac33a80bd5114aec74eb8,perfumaria,perfumery
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,e73b67b67587f7644d5bd1a52deb1b01,5.0,...,2018-08-22 19:07:58,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,17.0,On Time,aa4383b373c6aca5d8797843e5594415,automotivo,auto
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,359d03e676b3c069f62cadba8dd3f6e8,5.0,...,2017-12-05 19:21:58,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,12.0,On Time,d0b61bfb1de832b15ba9d266ca96e5b0,pet_shop,pet_shop
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,e50934924e227544ba8246aeb3770dd4,5.0,...,2018-02-18 13:02:51,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,9.0,On Time,65266b2da20d04dbe00c5c2d3bb7859e,papelaria,stationery


In [6]:
df.to_csv(os.path.join("../Merged_Olist_Data", "Last_Mile_Master_Sheet.csv"), index=False)
print(f" delivery_analysis.csv updated with english category name and saved as Last_Mile_Master_Sheet.csv")
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")

 delivery_analysis.csv updated with english category name and saved as Last_Mile_Master_Sheet.csv
Shape: (98207, 23)

Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'review_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'Days_Difference', 'Delivery_Status', 'product_id', 'product_category_name', 'product_category_name_english']


In [7]:
category_analysis = (
    df.groupby("product_category_name_english")
    .agg(
        total_orders=("order_id", "count"),
        avg_delay=("Days_Difference", "mean"),
        avg_review=("review_score", "mean")
    )
    .reset_index()
)

# Filter to categories with enough orders to be meaningful
category_analysis = category_analysis[category_analysis["total_orders"] >= 100]
category_analysis["avg_delay"] = category_analysis["avg_delay"].round(2)
category_analysis["avg_review"] = category_analysis["avg_review"].round(2)


category_analysis = category_analysis.sort_values("avg_delay", ascending=True)

print("TOP 5 CATEGORIES with LESS DELAY:")
print(category_analysis.head(5))
print("\nTOP 5 CATEGORIES BEST in DELAY:")
print(category_analysis.tail(5))

TOP 5 CATEGORIES with LESS DELAY:
   product_category_name_english  total_orders  avg_delay  avg_review
36                          food           443       8.68        4.28
47                  home_confort           375       8.86        3.88
4                          audio           345       9.01        3.85
33       fashion_underwear_beach           121       9.50        3.93
24                        drinks           293       9.95        4.18

TOP 5 CATEGORIES BEST in DELAY:
   product_category_name_english  total_orders  avg_delay  avg_review
54                  market_place           273      12.77        4.07
63              small_appliances           619      12.84        4.21
1               air_conditioning           250      12.89        4.07
31                 fashion_shoes           236      13.31        4.25
34               fixed_telephony           214      13.91        3.94


In [9]:
worst = category_analysis.head(5)
best = category_analysis.tail(5)

In [1]:
"""
plot_data = pd.concat([worst, best])

colors = ["green" if x in worst["product_category_name_english"].values
          else "red"
          for x in plot_data["product_category_name_english"]]

fig, ax = plt.subplots(figsize=(12, 8))
ax.barh(
    plot_data["product_category_name_english"],
    plot_data["avg_delay"],
    color=colors,
    edgecolor="white"
)

ax.set_xlabel("Average Days Early (Higher = Better)", fontsize=11)
ax.set_title(
    "Product Category Delivery Performance\nTop 5 Best vs Top 5 Worst",
    fontsize=13
)

# Add value labels
for i, val in enumerate(plot_data["avg_delay"]):
    ax.text(val + 0.1, i, f"{val:.1f} days", va="center", fontsize=9)

legend = [
    mpatches.Patch(color="green", label="Best Performing Categories"),
    mpatches.Patch(color="red", label="Worst Performing Categories")
]
ax.legend(handles=legend)

plt.tight_layout()
plt.savefig(os.path.join("../Charts", "category_delay_chart.png"), dpi=150)
plt.show()
print("Category delay chart saved")
"""

'\nplot_data = pd.concat([worst, best])\n\ncolors = ["green" if x in worst["product_category_name_english"].values\n          else "red"\n          for x in plot_data["product_category_name_english"]]\n\nfig, ax = plt.subplots(figsize=(12, 8))\nax.barh(\n    plot_data["product_category_name_english"],\n    plot_data["avg_delay"],\n    color=colors,\n    edgecolor="white"\n)\n\nax.set_xlabel("Average Days Early (Higher = Better)", fontsize=11)\nax.set_title(\n    "Product Category Delivery Performance\nTop 5 Best vs Top 5 Worst",\n    fontsize=13\n)\n\n# Add value labels\nfor i, val in enumerate(plot_data["avg_delay"]):\n    ax.text(val + 0.1, i, f"{val:.1f} days", va="center", fontsize=9)\n\nlegend = [\n    mpatches.Patch(color="green", label="Best Performing Categories"),\n    mpatches.Patch(color="red", label="Worst Performing Categories")\n]\nax.legend(handles=legend)\n\nplt.tight_layout()\nplt.savefig(os.path.join("../Charts", "category_delay_chart.png"), dpi=150)\nplt.show()\nprin